Import Python modules

In [1]:
import pandas as pd
import numpy as np
import glob

Read in counts data

In [2]:
# Read in data
counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)
counts_df = counts_df.replace('', np.nan)

# Convert codon_site to string to avoid issues with rows with semicolon-separated codon sites
counts_df['codon_site'] = counts_df['codon_site'].astype(str)

counts_df.head()

/tmp/ipykernel_3816209/1834990010.py:2: DtypeWarning: Columns (0: codon_position, 1: codon_site) have mixed types. Specify dtype option on import or set low_memory=False.
  counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,mut_class,mut_type,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate,exclude_from_mut_rate_analysis
0,557,A557C,A,C,PB1,2,186,GAA,GCA,E,...,nonsynonymous,AC,all,PB1,PB1_all,2271,swine,0.000440,0.0,False
1,1457,A1457C,A,C,PB1,2,486,AAA,ACA,K,...,nonsynonymous,AC,all,PB1,PB1_all,2271,swine,0.446059,0.0,False
2,184,A184C,A,C,NS1,1,62,AGA,CGA,R,...,synonymous,AC,all,NS,NS_all,835,all,0.009581,0.0,False
3,1223,A1223C,A,C,PB2,2,408,AAT,ACT,N,...,nonsynonymous,AC,all,PB2,PB2_all,2277,avian,0.000878,0.0,False
4,1223,A1223C,A,C,PB2,2,408,AAT,ACT,N,...,nonsynonymous,AC,all,PB2,PB2_all,2277,human,0.000878,0.0,False


Read in predicted rates from neutral model

In [3]:
predicted_rates_df = pd.read_csv(
    '../results/expected_rates.csv',
    keep_default_na=False
)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.200626
1,AC,HA,AAC,0.319609
2,AC,HA,AAG,0.248512
3,AC,HA,AAT,0.313680
4,AC,HA,CAA,0.177674


Compute expected counts and subset the data to mutations with at least X actual or expected counts

In [4]:
counts_cutoff = 10
# Merge counts with predicted rates; do NOT pre-filter on counts here. The
# cutoff is applied separately to the nt-level and AA-level aggregations
# below, so a mutation passes if its summed counts at the level being
# analyzed reach the cutoff. Pre-filtering at the row level would drop
# low-count codon-context rows that, when summed for a synonymous AA
# mutation with multiple nt paths, contribute meaningfully to the AA total
# (and would cause aa_fitness_effects.csv to disagree with the per-subset
# pipeline, which sums all nt rows before filtering).
actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
)
print("Number of merged nt-level rows (no count cutoff applied):", len(actual_expected_df))
actual_expected_df.head()

Number of merged nt-level rows (no count cutoff applied): 2092491


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate,exclude_from_mut_rate_analysis,predicted_rate,expected_count
0,557,A557C,A,C,PB1,2,186,GAA,GCA,E,...,all,PB1,PB1_all,2271,swine,0.000440,0.0,False,0.087405,0.000038
1,1457,A1457C,A,C,PB1,2,486,AAA,ACA,K,...,all,PB1,PB1_all,2271,swine,0.446059,0.0,False,0.200288,0.089340
2,184,A184C,A,C,NS1,1,62,AGA,CGA,R,...,all,NS,NS_all,835,all,0.009581,0.0,False,0.298324,0.002858
3,1223,A1223C,A,C,PB2,2,408,AAT,ACT,N,...,all,PB2,PB2_all,2277,avian,0.000878,0.0,False,0.298812,0.000262
4,1223,A1223C,A,C,PB2,2,408,AAT,ACT,N,...,all,PB2,PB2_all,2277,human,0.000878,0.0,False,0.298812,0.000262


Compute fitness effects of single nucleotide mutations.

In [5]:
pseudo_count = 0.5
groupby_cols = [
    'host', 'subtype', 'segment', 'gene', 'site', 'wt_nt', 'mut_nt', 'nt_mut',
    'mut_class'
]
nt_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .query("actual_count >= @counts_cutoff | expected_count >= @counts_cutoff")
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
nt_fitness_df.to_csv('../results/nt_fitness_effects.csv', index=False)
print("Number of unique nt muts with estimated fitness effects:", len(nt_fitness_df))
nt_fitness_df.head()

Number of unique nt muts with estimated fitness effects: 126560


,host,subtype,segment,gene,site,wt_nt,mut_nt,nt_mut,mut_class,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,2,T,A,T2A,nonsynonymous,0,21.431796,-3.781085
1,all,H1,HA,HA,2,T,C,T2C,nonsynonymous,0,64.316486,-4.864707
2,all,H1,HA,HA,2,T,G,T2G,nonsynonymous,0,13.596328,-3.339062
3,all,H1,HA,HA,3,G,A,G3A,nonsynonymous,0,149.094533,-5.701076
5,all,H1,HA,HA,3,G,T,G3T,nonsynonymous,0,29.234256,-4.085447


Compute average fitness effects of synonymous nucleotide mutations at a given site.

In [6]:
groupby_cols = ['host', 'subtype', 'segment', 'gene', 'site']
site_syn_fitness_df = (
    nt_fitness_df
    .query("mut_class == 'synonymous'")
    .groupby(groupby_cols, as_index=False)
    .agg({'delta_fitness': 'mean'})
)
site_syn_fitness_df.to_csv('../results/sitewise_synonymous_fitness_effects.csv', index=False)
print("Number of sites with estimated synonymous fitness effects:", len(site_syn_fitness_df))
site_syn_fitness_df.head()

Number of sites with estimated synonymous fitness effects: 20754


,host,subtype,segment,gene,site,delta_fitness
0,all,H1,HA,HA,6,0.238471
1,all,H1,HA,HA,9,-0.035125
2,all,H1,HA,HA,12,-0.390096
3,all,H1,HA,HA,13,-0.479983
4,all,H1,HA,HA,15,-0.912843


Compute fitness effects of amino-acid mutations, aggregating data across nucleotide mutations that result in the same amino-acid mutation.

In [7]:
groupby_cols = [
    'host', 'subtype', 'segment', 'gene', 'codon_site', 'wt_aa', 'mut_aa', 'aa_mut',
    'mut_class'
]
aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .query("actual_count >= @counts_cutoff | expected_count >= @counts_cutoff")
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
aa_fitness_df.to_csv('../results/aa_fitness_effects.csv', index=False)
print("Number of unique aa muts with estimated fitness effects:", len(aa_fitness_df))
aa_fitness_df.head()

Number of unique aa muts with estimated fitness effects: 110823


,host,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,1,M,I,M1I,nonsynonymous,0,179.500389,-5.886106
1,all,H1,HA,HA,1,M,K,M1K,nonsynonymous,0,21.431796,-3.781085
2,all,H1,HA,HA,1,M,R,M1R,nonsynonymous,0,13.596328,-3.339062
3,all,H1,HA,HA,1,M,T,M1T,nonsynonymous,0,64.316486,-4.864707
5,all,H1,HA,HA,10,C,C,C10C,synonymous,16,12.568988,0.233118


Report the number of amino-acid level mutations with data in each category (including "synonymous" mutations that don't change the amino acid)

In [8]:
aa_fitness_df.groupby(['host', 'mut_class']).size()

host   mut_class    
all    nonsense          2246
       nonsynonymous    33634
       synonymous        7587
avian  nonsense          1000
       nonsynonymous    15983
       synonymous        5459
human  nonsense          1834
       nonsynonymous    25358
       synonymous        5429
swine  nonsense           279
       nonsynonymous     8083
       synonymous        3931
dtype: int64

In [9]:
aa_fitness_df[
    (aa_fitness_df['host'] == 'all') &
    (aa_fitness_df['mut_class'] == 'nonsynonymous')
].groupby(['segment', 'subtype']).size()

segment  subtype
HA       H1         2847
         H3         2632
         H5         1519
         H7           32
         H9          917
MP       all        2094
NA       N1         2466
         N2         2484
         N6          362
         N8          364
         N9           16
NP       all        3115
NS       all        2041
PA       all        4266
PB1      all        4040
PB2      all        4439
dtype: int64